In [ ]:
import subprocess
subprocess.run(["pip", "install", "nltk"])
subprocess.run(["pip", "install", "spacy"])

import os
print("✅ Working directory:", os.getcwd())

In [ ]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
print("✅ NLTK data downloaded")

In [ ]:
# Always test on one file first
test_file = "data/processed/HDFC_Q1FY25.txt"

with open(test_file, "r", encoding="utf-8") as f:
    raw_text = f.read()

print(f"Total characters: {len(raw_text):,}")
print(f"\n--- First 2000 characters ---\n")
print(raw_text[:2000])

In [ ]:
import re

def extract_speakers(text):
    """
    Splits transcript text into individual speaker utterances.
    Looks for pattern: 'Speaker Name: dialogue text'
    """
    # Pattern: Line starting with Name (words + spaces): text
    pattern = r'^([A-Z][A-Za-z\s\.\-]+):\s*(.+?)(?=\n[A-Z][A-Za-z\s\.\-]+:|$)'
    
    matches = re.findall(pattern, text, re.MULTILINE | re.DOTALL)
    
    utterances = []
    for speaker, dialogue in matches:
        speaker = speaker.strip()
        dialogue = dialogue.strip().replace('\n', ' ')
        
        # Skip very short utterances (less than 10 words)
        if len(dialogue.split()) < 10:
            continue
            
        utterances.append({
            "speaker": speaker,
            "utterance": dialogue
        })
    
    return utterances

# Test it
utterances = extract_speakers(raw_text)
print(f"Total speaker turns found: {len(utterances)}")
print(f"\n--- First 3 utterances ---")
for u in utterances[:3]:
    print(f"\nSpeaker: {u['speaker']}")
    print(f"Text: {u['utterance'][:200]}...")

In [ ]:
def classify_role(speaker_name):
    """
    Tags each speaker as Management, Analyst, or Moderator
    """
    speaker_lower = speaker_name.lower()
    
    moderator_keywords = ['moderator', 'operator', 'coordinator', 'host']
    analyst_keywords = ['analyst', 'research', 'capital', 'securities', 
                       'asset', 'fund', 'investment', 'advisor', 'manager']
    
    if any(word in speaker_lower for word in moderator_keywords):
        return 'Moderator'
    elif any(word in speaker_lower for word in analyst_keywords):
        return 'Analyst'
    else:
        return 'Management'

# Test classification
for u in utterances[:5]:
    role = classify_role(u['speaker'])
    print(f"{u['speaker']} → {role}")

In [ ]:
def clean_text(text):
    """
    Removes boilerplate, extra spaces, and noise from transcript text
    """
    # Remove common boilerplate phrases
    boilerplate = [
        r'Classification\s*[–-]\s*Public',
        r'HDFC Bank Limited',
        r'Ladies and gentlemen',
        r'Thank you for joining',
        r'This is a transcription',
        r'Page \d+ of \d+',
        r'\[.*?\]',          # Remove anything in square brackets
        r'\(.*?inaudible.*?\)',  # Remove (inaudible) markers
    ]
    
    for pattern in boilerplate:
        text = re.sub(pattern, '', text, flags=re.IGNORECASE)
    
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

# Test it
sample = utterances[2]['utterance']
print("Before cleaning:")
print(sample[:300])
print("\nAfter cleaning:")
print(clean_text(sample)[:300])

In [ ]:
import pandas as pd

processed_folder = "data/processed"
all_utterances = []

for filename in os.listdir(processed_folder):
    if not filename.endswith(".txt"):
        continue
    
    # Get metadata from filename
    parts = filename.replace(".txt", "").split("_")
    ticker = parts[0]
    quarter = parts[1]
    
    filepath = os.path.join(processed_folder, filename)
    with open(filepath, "r", encoding="utf-8") as f:
        text = f.read()
    
    # Extract speakers
    utterances = extract_speakers(text)
    
    for u in utterances:
        cleaned = clean_text(u['utterance'])
        role = classify_role(u['speaker'])
        
        all_utterances.append({
            "ticker": ticker,
            "quarter": quarter,
            "speaker": u['speaker'],
            "role": role,
            "utterance": cleaned,
            "word_count": len(cleaned.split())
        })

# Create DataFrame
df = pd.DataFrame(all_utterances)

print(f"✅ Total utterances extracted: {len(df)}")
print(f"\nBy company:\n{df.groupby('ticker').size()}")
print(f"\nBy role:\n{df.groupby('role').size()}")
print(f"\nSample rows:")
print(df[['ticker', 'quarter', 'speaker', 'role', 'word_count']].head(10))

In [ ]:
output_path = "data/structured_utterances.csv"
df.to_csv(output_path, index=False)
print(f"✅ Saved {len(df)} utterances to {output_path}")

In [ ]:
def classify_role_v2(speaker_name):
    """
    Improved role classification for Indian earnings calls
    """
    speaker_lower = speaker_name.lower()
    
    # Known management titles
    management_keywords = [
        'ceo', 'cfo', 'coo', 'cto', 'chairman', 'director', 
        'president', 'managing', 'executive', 'officer',
        'jagdishan', 'vaidyanathan', 'lakhpatwala',  # HDFC names
        'kotak', 'birla', 'tata', 'chandrasekaran'   # Company heads
    ]
    
    moderator_keywords = ['moderator', 'operator', 'coordinator', 'sub']
    
    if any(word in speaker_lower for word in moderator_keywords):
        return 'Moderator'
    elif any(word in speaker_lower for word in management_keywords):
        return 'Management'
    else:
        # Default unknown speakers to Analyst 
        # (in earnings calls, external speakers are almost always analysts)
        return 'Analyst'

# Apply to existing dataframe
df['role'] = df['speaker'].apply(classify_role_v2)

print("Updated role breakdown:")
print(df.groupby('role').size())

In [ ]:
df.to_csv("data/structured_utterances.csv", index=False)
print(f"✅ Saved: {len(df)} utterances to data/structured_utterances.csv")

In [ ]:
import pandas as pd

df = pd.read_csv("data/structured_utterances.csv")

def classify_role_v3(speaker_name):
    speaker_lower = str(speaker_name).lower()
    
    moderator_keywords = ['moderator', 'operator', 'coordinator', 'sub', 'speakers']
    
    management_keywords = [
        # Generic titles
        'ceo', 'cfo', 'coo', 'cto', 'chairman', 'director',
        'president', 'managing', 'executive', 'officer',
        # HDFC names
        'jagdishan', 'vaidyanathan', 'lakhpatwala',
        # Tata names
        'chandrasekaran', 'padmanabhan', 'mistry',
        # HUL names — adding these now
        'jawa', 'tiwari', 'mulgaonkar', 'nair', 'srinivas',
        'mehta', 'khattar', 'gupta', 'rohit', 'ritesh', 'yogesh'
    ]
    
    if any(word in speaker_lower for word in moderator_keywords):
        return 'Moderator'
    elif any(word in speaker_lower for word in management_keywords):
        return 'Management'
    else:
        return 'Analyst'

# Apply updated classifier
df['role'] = df['speaker'].apply(classify_role_v3)

print("Updated role breakdown:")
print(df.groupby('role').size())
print("\nHUL role breakdown:")
print(df[df['ticker']=='HUL'].groupby('role').size())

# Save back
df.to_csv("data/structured_utterances.csv", index=False)
print("\n✅ Saved updated structured_utterances.csv")

In [ ]:
import os
import re
import pandas as pd
print("✅ Ready")

In [ ]:
def extract_speakers(text):
    pattern = r'^([A-Z][A-Za-z\s\.\-]+):\s*(.+?)(?=\n[A-Z][A-Za-z\s\.\-]+:|$)'
    matches = re.findall(pattern, text, re.MULTILINE | re.DOTALL)
    utterances = []
    for speaker, dialogue in matches:
        speaker = speaker.strip()
        dialogue = dialogue.strip().replace('\n', ' ')
        if len(dialogue.split()) < 10:
            continue
        utterances.append({"speaker": speaker, "utterance": dialogue})
    return utterances

def clean_text(text):
    boilerplate = [
        r'Classification\s*[–-]\s*Public',
        r'Page \d+ of \d+',
        r'\[.*?\]',
        r'\(.*?inaudible.*?\)',
    ]
    for pattern in boilerplate:
        text = re.sub(pattern, '', text, flags=re.IGNORECASE)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def classify_role(speaker_name):
    speaker_lower = str(speaker_name).lower()
    
    moderator_keywords = ['moderator', 'operator', 'coordinator', 'sub', 'speakers']
    
    management_keywords = [
        # Generic titles
        'ceo', 'cfo', 'coo', 'cto', 'chairman', 'director',
        'president', 'managing', 'executive', 'officer',
        # HDFC
        'jagdishan', 'vaidyanathan', 'lakhpatwala',
        # HUL
        'jawa', 'tiwari', 'mulgaonkar', 'nair', 'rohit', 'ritesh', 'yogesh',
        # Tata
        'chandrasekaran', 'padmanabhan',
        # Reliance
        'ambani', 'jain', 'srikanth', 'anshuman',
        # Wipro
        'delaporte', 'thierry', 'aparna',
        # Infosys
        'parekh', 'nilekani', 'salil', 'jayesh',
        # TCS
        'gopinathan', 'krithivasan', 'samir', 'milind',
        # Bajaj Finance
        'jain', 'rajeev', 'sandeep',
        # Maruti
        'bhargava', 'kenichi', 'hisashi', 'shashank',
        # Sun Pharma
        'shanghvi', 'dilip', 'sudhir', 'sailesh'
    ]
    
    if any(word in speaker_lower for word in moderator_keywords):
        return 'Moderator'
    elif any(word in speaker_lower for word in management_keywords):
        return 'Management'
    else:
        return 'Analyst'

print("✅ Functions defined")

In [ ]:
processed_folder = "data/processed"
all_utterances = []

company_info = {
    "HDFC":      ("HDFC Bank", "Banking"),
    "HUL":       ("Hindustan Unilever", "FMCG"),
    "Tata":      ("Tata Group", "Conglomerate"),
    "RELIANCE":  ("Reliance Industries", "Conglomerate"),
    "WIPRO":     ("Wipro", "IT"),
    "INFY":      ("Infosys", "IT"),
    "TCS":       ("TCS", "IT"),
    "BAJAJFIN":  ("Bajaj Finance", "NBFC"),
    "MARUTI":    ("Maruti Suzuki", "Auto"),
    "SUNPHARMA": ("Sun Pharma", "Pharma"),
}

for filename in sorted(os.listdir(processed_folder)):
    if not filename.endswith(".txt"):
        continue

    parts = filename.replace(".txt", "").split("_")
    if len(parts) < 2:
        continue

    ticker = parts[0]
    quarter = parts[1]
    company_name, sector = company_info.get(ticker, ("Unknown", "Unknown"))

    filepath = os.path.join(processed_folder, filename)
    with open(filepath, "r", encoding="utf-8") as f:
        text = f.read()

    utterances = extract_speakers(text)

    for u in utterances:
        cleaned = clean_text(u['utterance'])
        role = classify_role(u['speaker'])

        all_utterances.append({
            "ticker": ticker,
            "company": company_name,
            "sector": sector,
            "quarter": quarter,
            "speaker": u['speaker'],
            "role": role,
            "utterance": cleaned,
            "word_count": len(cleaned.split())
        })

df = pd.DataFrame(all_utterances)
df.to_csv("data/structured_utterances.csv", index=False)

print(f"✅ Total utterances: {len(df)}")
print(f"\nBy company:")
print(df.groupby('ticker').size())
print(f"\nBy role:")
print(df.groupby('role').size())

In [ ]:
import pandas as pd

df = pd.read_csv("data/structured_utterances.csv")

# Check what speakers exist for missing companies
for ticker in ['INFY', 'MARUTI', 'Reliance']:
    print(f"\n=== {ticker} unique speakers ===")
    speakers = df[df['ticker']==ticker]['speaker'].unique()
    for s in speakers[:15]:
        print(f"  '{s}'")

In [ ]:
def classify_role(speaker_name):
    speaker_lower = str(speaker_name).lower()
    
    moderator_keywords = ['moderator', 'operator', 'coordinator', 'sub', 'speakers']
    
    management_keywords = [
        # Generic titles
        'ceo', 'cfo', 'coo', 'cto', 'chairman', 'director',
        'president', 'managing', 'executive', 'officer',
        # HDFC
        'jagdishan', 'vaidyanathan', 'lakhpatwala',
        # HUL
        'jawa', 'tiwari', 'mulgaonkar', 'nair', 'rohit', 'ritesh', 'yogesh',
        # Tata
        'chandrasekaran', 'padmanabhan',
        # Wipro
        'delaporte', 'thierry', 'aparna',
        # Infosys
        'parekh', 'nilekani', 'salil', 'jayesh',
        # TCS
        'gopinathan', 'krithivasan', 'samir', 'milind',
        # Bajaj Finance
        'rajeev', 'sandeep',
        # Maruti — adding now
        'bhargava', 'kenichi', 'hisashi', 'shashank',
        'pranav', 'ambaprasad', 'binay', 'kumar rakesh',
        'hiranandani', 'amit',
        # Sun Pharma
        'shanghvi', 'dilip', 'sudhir', 'sailesh'
    ]
    
    if any(word in speaker_lower for word in moderator_keywords):
        return 'Moderator'
    elif any(word in speaker_lower for word in management_keywords):
        return 'Management'
    else:
        return 'Analyst'

print("✅ Updated classifier ready")

In [ ]:
import os

# Remove from raw_transcripts
raw_folder = "data/raw_transcripts"
processed_folder = "data/processed"

removed = []
for folder in [raw_folder, processed_folder]:
    for f in os.listdir(folder):
        if f.startswith("INFY") or f.startswith("Reliance"):
            os.remove(os.path.join(folder, f))
            removed.append(f)
            print(f"🗑️ Removed: {f}")

print(f"\n✅ Removed {len(removed)} files")
print(f"\nRemaining PDFs: {len([f for f in os.listdir(raw_folder) if f.endswith('.pdf')])}")

In [ ]:
import re
import pandas as pd

company_info = {
    "HDFC":      ("HDFC Bank", "Banking"),
    "HUL":       ("Hindustan Unilever", "FMCG"),
    "Tata":      ("Tata Group", "Conglomerate"),
    "WIPRO":     ("Wipro", "IT"),
    "TCS":       ("TCS", "IT"),
    "BAJAJFIN":  ("Bajaj Finance", "NBFC"),
    "MARUTI":    ("Maruti Suzuki", "Auto"),
    "SUNPHARMA": ("Sun Pharma", "Pharma"),
}

all_utterances = []

for filename in sorted(os.listdir(processed_folder)):
    if not filename.endswith(".txt"):
        continue
    if filename.startswith("INFY") or filename.startswith("Reliance"):
        continue

    parts = filename.replace(".txt", "").split("_")
    if len(parts) < 2:
        continue

    ticker = parts[0]
    quarter = parts[1]
    company_name, sector = company_info.get(ticker, ("Unknown", "Unknown"))

    with open(os.path.join(processed_folder, filename), "r", encoding="utf-8") as f:
        text = f.read()

    utterances = extract_speakers(text)

    for u in utterances:
        cleaned = clean_text(u['utterance'])
        role = classify_role(u['speaker'])
        all_utterances.append({
            "ticker": ticker,
            "company": company_name,
            "sector": sector,
            "quarter": quarter,
            "speaker": u['speaker'],
            "role": role,
            "utterance": cleaned,
            "word_count": len(cleaned.split())
        })

df = pd.DataFrame(all_utterances)
df.to_csv("data/structured_utterances.csv", index=False)

print(f"✅ Total utterances: {len(df)}")
print(f"\nBy company:")
print(df.groupby('ticker').size())
print(f"\nBy role:")
print(df.groupby('role').size())
print(f"\nManagement by company:")
print(df[df['role']=='Management'].groupby('ticker').size())